In [5]:
!pip install mlflow dagshub -q

## Setup and MLflow Initialization
We connect to DagsHub and set the experiment to **LinearModels_Training**. Every run in this notebook is grouped under that experiment so results are clearly separated from other architectures on DagsHub.

In [6]:
import pandas as pd
import numpy as np
import mlflow
import dagshub
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

# Connect to DagsHub
os.environ['MLFLOW_TRACKING_USERNAME'] = 'GigiSichinava'
dagshub.init(repo_owner='GigiSichinava', repo_name='ML-Assignment-2')
mlflow.set_tracking_uri("https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow")
mlflow.set_experiment("LinearModels_Training")

print("DagsHub connection established.")
print("Experiment: LinearModels_Training")


Initialized MLflow to track repo "GigiSichinava/ML-Assignment-2"

Repository GigiSichinava/ML-Assignment-2 initialized!

DagsHub connection established.
Experiment: LinearModels_Training


## Data Loading
We load both the transaction and identity tables then merge them on TransactionID with a left join. This keeps all transactions even those with no identity record.

In [7]:
# Load and merge the two tables on TransactionID
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity     = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

# Left join: keeps all transactions even those without an identity record
train_df = train_transaction.merge(train_identity, on='TransactionID', how='left')
test_df  = test_transaction.merge(test_identity,  on='TransactionID', how='left')

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"Fraud rate:  {train_df['isFraud'].mean():.4f}  ({train_df['isFraud'].sum()} fraud cases)")


Train shape: (590540, 434)
Test shape:  (506691, 433)
Fraud rate:  0.0350  (20663 fraud cases)


## Data Cleaning
Three cleaning steps are applied and logged as a single MLflow run. First we drop columns where more than 50% of values are missing. Then numeric NaNs are filled with the column median — robust to the heavy skew in financial data. Categorical NaNs are filled with the column mode.

In [8]:
with mlflow.start_run(run_name="LinearModels_Cleaning"):
    mlflow.log_param("stage", "Cleaning")

    # Separate target and drop the ID column
    y = train_df['isFraud'].copy()
    train_df.drop(columns=['isFraud', 'TransactionID'], inplace=True, errors='ignore')
    test_df.drop(columns=['TransactionID'], inplace=True, errors='ignore')

    # Step 1: Drop columns where more than 50% of values are missing
    # These columns carry too little information to be useful after imputation
    missing_ratio = train_df.isnull().mean()
    high_missing  = missing_ratio[missing_ratio > 0.5].index.tolist()
    train_df.drop(columns=high_missing, inplace=True)
    test_df.drop(columns=[c for c in high_missing if c in test_df.columns], inplace=True)
    print(f"Dropped {len(high_missing)} high-missing columns")

    # Identify numeric and categorical columns after dropping
    num_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = train_df.select_dtypes(include=['object']).columns.tolist()

    # Step 2: Fill numeric NaN with median
    # Median is more robust than mean because transaction amounts are heavily skewed
    for col in num_cols:
        median_val = train_df[col].median()
        train_df[col] = train_df[col].fillna(median_val)
        test_df[col]  = test_df[col].fillna(median_val)

    # Step 3: Fill categorical NaN with mode
    # The most frequent category is the safest default for unknown values
    for col in cat_cols:
        mode_val = train_df[col].mode()[0]
        train_df[col] = train_df[col].fillna(mode_val)
        test_df[col]  = test_df[col].fillna(mode_val)

    mlflow.log_param("dropped_columns",   len(high_missing))
    mlflow.log_param("numeric_features",  len(num_cols))
    mlflow.log_param("categorical_features", len(cat_cols))
    print(f"After cleaning: Train={train_df.shape}, Test={test_df.shape}")
    print("Cleaning run logged to MLflow.")


Dropped 214 high-missing columns
After cleaning: Train=(590540, 218), Test=(506691, 256)
Cleaning run logged to MLflow.
🏃 View run LinearModels_Cleaning at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/3/runs/212111bcf48d453b94e249dff1180f78
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/3


## Feature Engineering
We create five new features that capture fraud signals not visible in the raw columns. Log-transformed amount corrects for skew. The decimal part captures suspicious cent patterns. Hour and day features expose temporal fraud behaviour. All categorical columns are label-encoded since tree models handle integer inputs well.

In [9]:
# Feature 1: Log transform of transaction amount
# Transaction amounts are right skewed so log1p brings the distribution closer to normal
train_df['TransactionAmt_log'] = np.log1p(train_df['TransactionAmt'])
test_df['TransactionAmt_log']  = np.log1p(test_df['TransactionAmt'])

# Feature 2: Decimal part of transaction amount
# Fraudsters often use amounts like 99.00 or 0.01 — the cents part captures this
train_df['TransactionAmt_decimal'] = train_df['TransactionAmt'] - train_df['TransactionAmt'].astype(int)
test_df['TransactionAmt_decimal']  = test_df['TransactionAmt'] - test_df['TransactionAmt'].astype(int)

# Feature 3: Hour of day extracted from TransactionDT timestamp
# Fraud tends to happen at unusual hours like late night
train_df['Transaction_hour'] = (train_df['TransactionDT'] // 3600) % 24
test_df['Transaction_hour']  = (test_df['TransactionDT']  // 3600) % 24

# Feature 4: Day of week extracted from TransactionDT
# Fraud patterns often differ by day of week
train_df['Transaction_day'] = (train_df['TransactionDT'] // (3600 * 24)) % 7
test_df['Transaction_day']  = (test_df['TransactionDT']  // (3600 * 24)) % 7

# Feature 5: Label encode all categorical columns
# Tree models handle integer encoded categories well without one-hot expansion
le = LabelEncoder()
for col in cat_cols:
    if col in train_df.columns:
        combined = pd.concat([train_df[col], test_df[col]], axis=0).astype(str)
        le.fit(combined)
        train_df[col] = le.transform(train_df[col].astype(str))
        test_df[col]  = le.transform(test_df[col].astype(str))

print(f"Feature engineering complete. Final shape: {train_df.shape}")


/tmp/ipykernel_57/1320925072.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df['TransactionAmt_log'] = np.log1p(train_df['TransactionAmt'])
/tmp/ipykernel_57/1320925072.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df['TransactionAmt_log']  = np.log1p(test_df['TransactionAmt'])
/tmp/ipykernel_57/1320925072.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.conca

Feature engineering complete. Final shape: (590540, 222)


## Feature Selection
Three sequential methods are applied and logged to MLflow. Variance threshold removes near-constant columns. Correlation filter removes redundant pairs above 0.95. Random Forest importance then selects the top 50 most predictive features. The train/validation split is defined here and used by all training cells below.

In [10]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="LinearModels_Feature_Selection"):
    mlflow.log_param("stage", "Feature_Selection")

    X         = train_df.copy()
    X_test_fs = test_df.copy()

    # Method 1: Variance threshold removes near-constant columns
    vt           = VarianceThreshold(threshold=0.01)
    vt.fit(X)
    low_var_cols = X.columns[~vt.get_support()].tolist()
    X            = X.drop(columns=low_var_cols)
    X_test_fs    = X_test_fs.drop(columns=[c for c in low_var_cols if c in X_test_fs.columns])
    print(f"After variance threshold: {X.shape[1]} features")

    # Method 2: Correlation filter removes redundant feature pairs above 0.95
    corr_matrix    = X.corr().abs()
    upper          = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    high_corr_cols = [col for col in upper.columns if any(upper[col] > 0.95)]
    X              = X.drop(columns=high_corr_cols)
    X_test_fs      = X_test_fs.drop(columns=[c for c in high_corr_cols if c in X_test_fs.columns])
    print(f"After correlation filter: {X.shape[1]} features")

    # Method 3: Random Forest importance keeps the top 50 most predictive features
    rf_selector  = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1)
    rf_selector.fit(X, y)
    importances  = pd.Series(rf_selector.feature_importances_, index=X.columns)
    top_features = importances.nlargest(50).index.tolist()

    mlflow.log_param("features_after_variance", X.shape[1])
    mlflow.log_param("features_after_corr",     X.shape[1] - len(high_corr_cols))
    mlflow.log_param("final_feature_count",      len(top_features))
    print(f"Top 50 features selected by RF importance")
    print("Feature selection run logged to MLflow.")

# Define train/val split outside the mlflow block so variables are globally available
X_selected      = X[top_features]
X_test_selected = X_test_fs[top_features]

# Stratified split: preserves the fraud ratio in both train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_selected, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Val: {X_val.shape}")


After variance threshold: 199 features
After correlation filter: 144 features
Top 50 features selected by RF importance
Feature selection run logged to MLflow.
🏃 View run LinearModels_Feature_Selection at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/3/runs/0f0b0802f214404b805cf1e64fec40c7
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/3
Train: (472432, 50), Val: (118108, 50)


## Model Training
We test Ridge Classifier with six different alpha values to observe underfitting and overfitting. Very high alpha causes underfitting by forcing all weights toward zero. Very low alpha risks overfitting by applying almost no regularization. Each run is logged to MLflow under the LinearModels Training experiment.

In [11]:
from sklearn.linear_model import RidgeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score

results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Ridge uses L2 regularization — higher alpha means stronger regularization
# Very high alpha forces all weights toward zero which causes underfitting
# Very low alpha means almost no regularization which risks overfitting
ridge_alphas = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

print("=== Ridge Classifier alpha sweep ===")
for alpha in ridge_alphas:
    label = f"Ridge_alpha_{alpha}"
    model = RidgeClassifier(alpha=alpha)
    with mlflow.start_run(run_name=label):
        # Note: RidgeClassifier uses decision_function instead of predict_proba
        cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
        model.fit(X_train, y_train)
        train_auc = roc_auc_score(y_train, model.decision_function(X_train))
        val_auc   = roc_auc_score(y_val,   model.decision_function(X_val))
        gap       = train_auc - val_auc

        mlflow.log_param("model_type", label)
        mlflow.log_param("alpha",      alpha)
        mlflow.log_metric("train_auc",   train_auc)
        mlflow.log_metric("val_auc",     val_auc)
        mlflow.log_metric("cv_auc_mean", cv_scores.mean())
        mlflow.log_metric("cv_auc_std",  cv_scores.std())
        mlflow.log_metric("overfit_gap", gap)

        results.append({"label": label, "train_auc": train_auc, "val_auc": val_auc, "gap": gap})
        print(f"{label}: Train={train_auc:.4f} Val={val_auc:.4f} Gap={gap:.4f}")
        if alpha >= 100.0:
            print("   Note: Very high alpha — strong regularization, underfitting risk.")
        if alpha <= 0.01:
            print("   Note: Very low alpha — weak regularization, overfitting risk.")
        print("-" * 30)

print("\nRidge training complete.")


=== Ridge Classifier alpha sweep ===
Ridge_alpha_0.01: Train=0.7850 Val=0.7841 Gap=0.0009
   Note: Very low alpha — weak regularization, overfitting risk.
------------------------------
🏃 View run Ridge_alpha_0.01 at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/3/runs/eeada64881714dd69a2086736b83c377
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/3
Ridge_alpha_0.1: Train=0.7850 Val=0.7841 Gap=0.0009
------------------------------
🏃 View run Ridge_alpha_0.1 at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/3/runs/0bb6c02a89d24d2090df43f4f7b8c7d6
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/3
Ridge_alpha_1.0: Train=0.7850 Val=0.7841 Gap=0.0009
------------------------------
🏃 View run Ridge_alpha_1.0 at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/3/runs/217fcf2d68c648ad8e4c82c0b437da61
🧪 View experiment at: https://d

## Best Pipeline Registration
The best Ridge configuration is wrapped in a sklearn Pipeline together with a preprocessing function. This pipeline runs directly on raw unprocessed data because it applies all cleaning and feature engineering steps internally. The pipeline is saved to the DagsHub Model Registry.

In [12]:
from sklearn.linear_model import RidgeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, LabelEncoder
import numpy as np

def preprocess_raw(df):
    df = df.copy()

    # Preprocessing step: drop high missing columns using the same 50% threshold
    missing_ratio = df.isnull().mean()
    df = df.drop(columns=missing_ratio[missing_ratio > 0.5].index.tolist(), errors='ignore')

    num_c = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_c = df.select_dtypes(include=['object']).columns.tolist()

    # Fill missing values: same strategy as in the Cleaning section
    for col in num_c:
        df[col] = df[col].fillna(df[col].median())
    for col in cat_c:
        df[col] = df[col].fillna(df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown')

    # Apply engineered features: same ones created in the Feature Engineering section
    df['TransactionAmt_log']     = np.log1p(df['TransactionAmt'])
    df['TransactionAmt_decimal'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
    df['Transaction_hour']       = (df['TransactionDT'] // 3600) % 24
    df['Transaction_day']        = (df['TransactionDT'] // (3600 * 24)) % 7

    # Encode categoricals
    le = LabelEncoder()
    for col in cat_c:
        if col in df.columns:
            df[col] = le.fit_transform(df[col].astype(str))

    # Keep only top features: the ones selected during Feature Selection
    keep = [c for c in top_features if c in df.columns]
    return df[keep]

best_ridge = RidgeClassifier(alpha=1.0)

ridge_pipeline = Pipeline([
    ('preprocessor', FunctionTransformer(preprocess_raw)),
    ('classifier',   best_ridge)
])

# Fit on raw data: the pipeline handles all preprocessing internally
train_raw = train_transaction.merge(train_identity, on='TransactionID', how='left')
y_raw     = train_raw['isFraud'].copy()
train_raw = train_raw.drop(columns=['isFraud', 'TransactionID'], errors='ignore')

ridge_pipeline.fit(train_raw, y_raw)
print("Ridge Pipeline trained on raw data successfully.")

# Register to Model Registry: save the pipeline for later use in inference
with mlflow.start_run(run_name="Ridge_Pipeline_Registration"):
    mlflow.log_param("model_type", "Ridge_Pipeline")
    mlflow.log_param("best_alpha", 1.0)
    mlflow.sklearn.log_model(
        sk_model=ridge_pipeline,
        artifact_path="model",
        registered_model_name="Ridge_Pipeline"
    )
    print("Ridge Pipeline registered in DagsHub Model Registry.")


/tmp/ipykernel_57/656985992.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['TransactionAmt_log']     = np.log1p(df['TransactionAmt'])
/tmp/ipykernel_57/656985992.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['TransactionAmt_decimal'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
/tmp/ipykernel_57/656985992.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once usin

Ridge Pipeline trained on raw data successfully.


2026/05/03 22:39:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 22:39:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'Ridge_Pipeline'.
2026/05/03 22:39:31 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Ridge_Pipeline, version 1
Created version '1' of model 'Ridge_Pipeline'.


Ridge Pipeline registered in DagsHub Model Registry.
🏃 View run Ridge_Pipeline_Registration at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/3/runs/3c23b9d71a2a4558b9be5414f5c2ce5b
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/3
